In [2]:
SELECT * FROM silver_patients

StatementMeta(, c594a050-c0bc-452f-9bc1-1d5f7ea22103, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 9 fields>

In [3]:
%%sql

CREATE TABLE dim_patient AS
SELECT
    PatientID,
    PatientName,
    DateOfBirth,
    Age,
    Gender,
    City,
    State,
    InsuranceType,
    RegistrationDate
FROM silver_patients;

StatementMeta(, c594a050-c0bc-452f-9bc1-1d5f7ea22103, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
%%sql

CREATE TABLE dim_doctor AS
SELECT
    DoctorID,
    DoctorName,
    DepartmentID,
    Specialization,
    EmploymentType,
    ExperienceYears
FROM silver_doctors;


StatementMeta(, c594a050-c0bc-452f-9bc1-1d5f7ea22103, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
%%sql

CREATE TABLE dim_hospital AS
SELECT
    HospitalID,
    HospitalName,
    HospitalType,
    City,
    State,
    TotalBeds
FROM silver_hospitals;

StatementMeta(, c594a050-c0bc-452f-9bc1-1d5f7ea22103, 8, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [9]:
%%sql

CREATE TABLE dim_department AS
SELECT
    DepartmentID,
    DepartmentName,
    HospitalID,
    DepartmentType
FROM silver_departments;

StatementMeta(, c594a050-c0bc-452f-9bc1-1d5f7ea22103, 10, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [10]:
%%sql

CREATE TABLE dim_date AS
SELECT
    DateKey,
    Date,
    YEAR(Date) AS Year,
    QUARTER(Date) AS Quarter,
    MONTH(Date) AS Month,
    DATE_FORMAT(Date, 'MMMM') AS MonthName,
    WEEKOFYEAR(Date) AS Week,
    DAY(Date) AS Day,
    DATE_FORMAT(Date, 'EEEE') AS DayName
FROM (
    SELECT
        CAST(DATE_FORMAT(Date, 'yyyyMMdd') AS INT) AS DateKey,
        Date
    FROM (
        SELECT EXPLODE(
            SEQUENCE(
                TO_DATE('2020-01-01'),
                TO_DATE('2026-12-31'),
                INTERVAL 1 DAY
            )
        ) AS Date
    )
);

StatementMeta(, c594a050-c0bc-452f-9bc1-1d5f7ea22103, 11, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [12]:
%%sql

CREATE TABLE fact_appointments AS
SELECT
    a.AppointmentID,
    a.PatientID,
    a.DoctorID,
    CAST(DATE_FORMAT(a.AppointmentDate, 'yyyyMMdd') AS INT) AS DateKey,
    a.AppointmentType,
    a.AppointmentStatus,
    a.WaitingTimeMinutes,
    a.WaitingTimeCategory,
    a.IsNoShow,
    a.IsCancelled
FROM silver_appointments a;

StatementMeta(, c594a050-c0bc-452f-9bc1-1d5f7ea22103, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [14]:

CREATE TABLE fact_admissions AS
SELECT
    AdmissionID,
    PatientID,
    HospitalID,
    DepartmentID,

    CAST(DATE_FORMAT(AdmissionDate, 'yyyyMMdd') AS INT) AS AdmissionDateKey,

    CAST(
        CASE
            WHEN DischargeDate IS NOT NULL
            THEN DATE_FORMAT(DischargeDate, 'yyyyMMdd')
        END AS INT
    ) AS DischargeDateKey,

    AdmissionDate,
    DischargeDate,

    AdmissionType,
    DischargeStatus,
    LengthOfStay,
    LOSCategory,
    PreviousAdmissionDate,
    DaysSincePreviousAdmission,
    IsReadmission

FROM silver_admissions;

StatementMeta(, c594a050-c0bc-452f-9bc1-1d5f7ea22103, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [17]:
CREATE TABLE fact_treatments AS
SELECT
    TreatmentID,
    PatientID,
    DoctorID,
    CAST(DATE_FORMAT(TreatmentDate, 'yyyyMMdd') AS INT) AS DateKey,
    TreatmentDate,
    TreatmentType,
    TreatmentCost,
    CostCategory,
    IsNegativeCost,
    IsZeroCost
FROM silver_treatments;

StatementMeta(, c594a050-c0bc-452f-9bc1-1d5f7ea22103, 18, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [19]:
%%sql

CREATE TABLE fact_bed_utilization AS
SELECT
    HospitalID,
    CAST(DATE_FORMAT(Date, 'yyyyMMdd') AS INT) AS DateKey,
    Date,
    TotalBeds,
    OccupiedBeds,
    AvailableBeds,
    OccupancyRate,
    OccupancyCategory
FROM silver_bed_utilization;

StatementMeta(, c594a050-c0bc-452f-9bc1-1d5f7ea22103, 20, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [22]:
%%sql

CREATE TABLE fact_patient_feedback AS
SELECT
    FeedbackID,
    PatientID,
    DepartmentID,
    CAST(DATE_FORMAT(FeedbackDate, 'yyyyMMdd') AS INT) AS DateKey,
    FeedbackDate,
    Rating,
    RatingCategory,
    FeedbackCategory
FROM silver_patient_feedback;

StatementMeta(, c594a050-c0bc-452f-9bc1-1d5f7ea22103, 23, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [24]:
%%sql

CREATE TABLE fact_insurance_claims AS
SELECT
    ClaimID,
    PatientID,
    CAST(DATE_FORMAT(ClaimDate, 'yyyyMMdd') AS INT) AS DateKey,
    ClaimDate,
    ClaimAmount,
    ApprovedAmount,
    ClaimApprovalPercentage,
    ClaimStatus,
    ClaimAmountCategory
FROM silver_insurance_claims;

StatementMeta(, c594a050-c0bc-452f-9bc1-1d5f7ea22103, 25, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [25]:
%%sql

SELECT 'silver_patients' AS TableName, COUNT(*) AS RowCount FROM silver_patients
UNION ALL
SELECT 'dim_patient', COUNT(*) FROM dim_patient
UNION ALL
SELECT 'silver_doctors', COUNT(*) FROM silver_doctors
UNION ALL
SELECT 'dim_doctor', COUNT(*) FROM dim_doctor
UNION ALL
SELECT 'silver_hospitals', COUNT(*) FROM silver_hospitals
UNION ALL
SELECT 'dim_hospital', COUNT(*) FROM dim_hospital
UNION ALL
SELECT 'silver_departments', COUNT(*) FROM silver_departments
UNION ALL
SELECT 'dim_department', COUNT(*) FROM dim_department
UNION ALL
SELECT 'silver_appointments', COUNT(*) FROM silver_appointments
UNION ALL
SELECT 'fact_appointments', COUNT(*) FROM fact_appointments
UNION ALL
SELECT 'silver_admissions', COUNT(*) FROM silver_admissions
UNION ALL
SELECT 'fact_admissions', COUNT(*) FROM fact_admissions
UNION ALL
SELECT 'silver_treatments', COUNT(*) FROM silver_treatments
UNION ALL
SELECT 'fact_treatments', COUNT(*) FROM fact_treatments
UNION ALL
SELECT 'silver_bed_utilization', COUNT(*) FROM silver_bed_utilization
UNION ALL
SELECT 'fact_bed_utilization', COUNT(*) FROM fact_bed_utilization
UNION ALL
SELECT 'silver_patient_feedback', COUNT(*) FROM silver_patient_feedback
UNION ALL
SELECT 'fact_patient_feedback', COUNT(*) FROM fact_patient_feedback
UNION ALL
SELECT 'silver_insurance_claims', COUNT(*) FROM silver_insurance_claims
UNION ALL
SELECT 'fact_insurance_claims', COUNT(*) FROM fact_insurance_claims;

StatementMeta(, c594a050-c0bc-452f-9bc1-1d5f7ea22103, 26, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 2 fields>

In [1]:
%%sql

CREATE TABLE kpi_hospital_performance AS

SELECT
    h.HospitalID,
    h.HospitalName,
    h.TotalBeds,

    COALESCE(a.TotalAdmissions, 0) AS TotalAdmissions,

    COALESCE(a.AverageLengthOfStay, 0) AS AverageLengthOfStay,

    COALESCE(a.ReadmissionRate, 0) AS ReadmissionRate,

    COALESCE(b.AverageOccupancyRate, 0) AS AverageOccupancyRate

FROM dim_hospital h

LEFT JOIN
(
    SELECT
        HospitalID,
        COUNT(*) AS TotalAdmissions,
        ROUND(AVG(LengthOfStay), 2) AS AverageLengthOfStay,
        ROUND(
            SUM(CASE WHEN IsReadmission = 'Yes' THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*),
            2
        ) AS ReadmissionRate
    FROM fact_admissions
    GROUP BY HospitalID
) a
    ON h.HospitalID = a.HospitalID

LEFT JOIN
(
    SELECT
        HospitalID,
        ROUND(AVG(OccupancyRate), 2) AS AverageOccupancyRate
    FROM fact_bed_utilization
    GROUP BY HospitalID
) b
    ON h.HospitalID = b.HospitalID;

StatementMeta(, 5e054509-4279-46cf-8c0b-59d20bb8ff99, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
%%sql
CREATE TABLE kpi_appointment_performance AS
SELECT
    a.DateKey,
    d.Date,
    COUNT(*) AS TotalAppointments,

    ROUND(
        SUM(CASE WHEN a.IsNoShow = 'Yes' THEN 1 ELSE 0 END)
        * 100.0 / COUNT(*),
        2
    ) AS NoShowRate,

    ROUND(
        SUM(CASE WHEN a.IsCancelled = 'Yes' THEN 1 ELSE 0 END)
        * 100.0 / COUNT(*),
        2
    ) AS CancellationRate,

    ROUND(
        AVG(a.WaitingTimeMinutes),
        2
    ) AS AverageWaitingTime

FROM fact_appointments a

LEFT JOIN dim_date d
    ON a.DateKey = d.DateKey

GROUP BY
    a.DateKey,
    d.Date;

StatementMeta(, 5e054509-4279-46cf-8c0b-59d20bb8ff99, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
%%sql
CREATE TABLE kpi_department_performance AS

SELECT
    d.DepartmentID,
    d.DepartmentName,

    COALESCE(a.TotalAdmissions, 0) AS TotalAdmissions,
    COALESCE(t.TotalTreatments, 0) AS TotalTreatments,
    COALESCE(t.TotalTreatmentCost, 0) AS TotalTreatmentCost,
    COALESCE(f.AveragePatientRating, 0) AS AveragePatientRating

FROM dim_department d

LEFT JOIN
(
    SELECT
        DepartmentID,
        COUNT(*) AS TotalAdmissions
    FROM fact_admissions
    GROUP BY DepartmentID
) a
    ON d.DepartmentID = a.DepartmentID

LEFT JOIN
(
    SELECT
        doc.DepartmentID,
        COUNT(*) AS TotalTreatments,
        ROUND(SUM(t.TreatmentCost), 2) AS TotalTreatmentCost
    FROM fact_treatments t
    JOIN dim_doctor doc
        ON t.DoctorID = doc.DoctorID
    GROUP BY doc.DepartmentID
) t
    ON d.DepartmentID = t.DepartmentID

LEFT JOIN
(
    SELECT
        DepartmentID,
        ROUND(AVG(Rating), 2) AS AveragePatientRating
    FROM fact_patient_feedback
    GROUP BY DepartmentID
) f
    ON d.DepartmentID = f.DepartmentID;

StatementMeta(, 5e054509-4279-46cf-8c0b-59d20bb8ff99, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
%%sql
CREATE TABLE kpi_insurance_performance AS

SELECT
    p.InsuranceType,
    COUNT(c.ClaimID) AS TotalClaims,
    ROUND(SUM(c.ClaimAmount), 2) AS ClaimAmount,
    ROUND(SUM(c.ApprovedAmount), 2) AS ApprovedAmount,
    ROUND(
        SUM(c.ApprovedAmount) * 100.0 /
        NULLIF(SUM(c.ClaimAmount), 0),
        2
    ) AS ApprovalRate

FROM fact_insurance_claims c

JOIN dim_patient p
    ON c.PatientID = p.PatientID

GROUP BY
    p.InsuranceType;

StatementMeta(, 5e054509-4279-46cf-8c0b-59d20bb8ff99, 7, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
%%sql
CREATE TABLE kpi_patient_analytics AS

SELECT
    p.RegistrationDate AS Date,
    YEAR(p.RegistrationDate) AS Year,
    MONTH(p.RegistrationDate) AS Month,
    COUNT(*) AS NewPatients,
    COUNT(DISTINCT p.PatientID) AS PatientCount,
    p.InsuranceType,
    COUNT(*) AS InsuranceMix

FROM dim_patient p

GROUP BY
    p.RegistrationDate,
    YEAR(p.RegistrationDate),
    MONTH(p.RegistrationDate),
    p.InsuranceType;

StatementMeta(, 5e054509-4279-46cf-8c0b-59d20bb8ff99, 8, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>